# Exercise 2 — Random Forest Classifier: Rock or Mine 🪨💣

**Dataset:** [Sonar — Rock or Mine Classification](https://www.kaggle.com/datasets/vijayaadithyanvg/rock-or-mine-classification)

The dataset contains **208 samples** of sonar signals bounced off either a **rock (R)** or a **metal cylinder / mine (M)**. Each sample has **60 numerical features** (energy in different frequency bands).

**Your task:** build a Random Forest classifier that predicts whether an object is a rock or a mine.

**Instructions:** Fill in every `___` gap, then run the cells.

---

## 0 · Install helper package (run once)

In [ ]:
# OpenML is used here so the notebook runs without the UCI ML Repo API.
# (Original exercise used: pip install ucimlrepo)


## 1 · Imports

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
)

np.random.seed(42)


## 2 · Load the data

We load the dataset directly from the **UCI Machine Learning Repository** (dataset id 151).
The features are 60 sonar frequency-band energy readings; the target is `"R"` (rock) or `"M"` (mine).

> **Alternative:** if you downloaded the CSV from Kaggle, replace the cell below with:
> ```python
> df = pd.read_csv("sonar_data.csv", header=None)
> df.rename(columns={60: "target"}, inplace=True)
> ```


In [ ]:
from sklearn.datasets import fetch_openml

sonar = fetch_openml(data_id=40, as_frame=True, parser="auto")
X_raw = sonar.data
y_labels = sonar.target.map({"Mine": "M", "Rock": "R"})

df = pd.concat([X_raw.reset_index(drop=True), y_labels.rename("target")], axis=1)
df.columns = [*range(60), "target"]

print(f"Shape: {df.shape}")
df.head()


## 3 · Quick exploratory look

In [ ]:
print(df["target"].value_counts())
print()
print(df.describe().T.head(10))


In [ ]:
# Visualise a few feature distributions by class
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
for i, col in enumerate([0, 10, 30]):
    for label, color in zip(["M", "R"], ["tomato", "steelblue"]):
        axes[i].hist(
            df.loc[df["target"] == label, col],
            bins=15, alpha=0.6, label=label, color=color,
        )
    axes[i].set_title(f"Feature {col}")
    axes[i].legend()
plt.tight_layout()
plt.show()

## 4 · Prepare features and target

Separate the 60 sonar features (`X`) from the label column (`y`).


In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("X shape:", X.shape)
print("y shape:", y.shape)


## 5 · Train / test split

Split the data — use **80 % train, 20 % test**, `random_state=42`, and **stratify** on `y`.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

print("Train:", X_train.shape, "| Test:", X_test.shape)


## 6 · Train a Random Forest

Create a `RandomForestClassifier` and fit it on the training set.

Suggested starting hyper-parameters:
- `n_estimators=100`
- `random_state=42`


In [ ]:
rf = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
)
rf.fit(X_train, y_train)


## 7 · Make predictions

In [ ]:
y_pred = rf.predict(X_test)


## 8 · Evaluate the model

Print the **accuracy** and the full **classification report**, then display a **confusion matrix**.


In [ ]:
acc = accuracy_score(y_test, y_pred)
print(f"Accuracy: {acc:.4f}\n")
print(classification_report(y_test, y_pred))


In [ ]:
cm = confusion_matrix(y_test, y_pred, labels=["M", "R"])
disp = ConfusionMatrixDisplay(cm, display_labels=["Mine", "Rock"])
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.show()


## 9 · Feature importance

Random Forest gives us feature importances for free. Plot the **top 15** most important features.


In [ ]:
importances = pd.Series(rf.feature_importances_, index=X.columns)
top15 = importances.nlargest(15)

plt.figure(figsize=(8, 5))
top15.sort_values().plot.barh(color="steelblue")
plt.xlabel("Importance")
plt.title("Top 15 Feature Importances")
plt.tight_layout()
plt.show()


## 10 · Bonus — Experiment 🧪

Try changing `n_estimators` or `max_depth` and see if you can improve the accuracy.


In [ ]:
rf2 = RandomForestClassifier(
    n_estimators=300,
    max_depth=8,
    random_state=42,
)
rf2.fit(X_train, y_train)
y_pred2 = rf2.predict(X_test)
print(f"New accuracy: {accuracy_score(y_test, y_pred2):.4f}")


---
✅ **Done!** You've trained and evaluated a Random Forest on the Sonar dataset.